# 05 - Assistants API met Vector Stores (Nederlandse Editie)

Deze notebook demonstreert de Assistants API met Vector Stores voor document-gebaseerde intelligentie, met voorbeelden relevant voor Nederland.

## 🇳🇱 Nederlandse Context
We bouwen een assistent die Nederlandse documenten kan doorzoeken:
- Nederlandse bedrijfsdocumenten
- AVG/GDPR compliance documenten
- Nederlandse wet- en regelgeving

## 🎯 Leerdoelen
- Assistants API begrijpen en implementeren
- Vector Stores voor document zoeken gebruiken
- Nederlandse documenten indexeren en doorzoeken

In [ ]:
import os
import json
import time
from openai import AzureOpenAI
from dotenv import load_dotenv
load_dotenv()

# Azure OpenAI Configuratie
client = AzureOpenAI(
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version="2024-05-01-preview"
)

deployment = os.getenv("AZURE_OPENAI_GPT4O_DEPLOYMENT_NAME")

print("✅ Azure OpenAI client geïnitialiseerd voor Assistants API")

## 📄 Stap 1: Maak Nederlandse Documenten

Laten we voorbeelddocumenten maken die relevant zijn voor Nederlandse bedrijven:

In [ ]:
# Nederlands AVG/GDPR document
avg_document = """
# AVG Compliance Handleiding voor Nederlandse Bedrijven

## Inleiding
De Algemene Verordening Gegevensbescherming (AVG) is sinds 25 mei 2018 van kracht in Nederland en de rest van de EU.

## Belangrijkste Principes

### 1. Rechtmatigheid, behoorlijkheid en transparantie
Persoonsgegevens moeten op een rechtmatige, behoorlijke en transparante wijze worden verwerkt.

### 2. Doelbinding
Gegevens mogen alleen worden verzameld voor welbepaalde, uitdrukkelijk omschreven en gerechtvaardigde doeleinden.

### 3. Minimale gegevensverwerking
Verzamel alleen de gegevens die noodzakelijk zijn voor het doel.

### 4. Juistheid
Persoonsgegevens moeten juist zijn en zo nodig worden bijgewerkt.

### 5. Opslagbeperking
Bewaar gegevens niet langer dan noodzakelijk.

### 6. Integriteit en vertrouwelijkheid
Zorg voor passende beveiliging van persoonsgegevens.

## Rechten van Betrokkenen
- Recht op inzage
- Recht op rectificatie
- Recht op vergetelheid (wissen)
- Recht op beperking van verwerking
- Recht op dataportabiliteit
- Recht van bezwaar

## Autoriteit Persoonsgegevens
De Autoriteit Persoonsgegevens (AP) is de Nederlandse toezichthouder voor privacy en AVG compliance.
Website: autoriteitpersoonsgegevens.nl
"""

# Nederlands personeelsbeleid document
hr_document = """
# Personeelshandboek TechBedrijf Nederland B.V.

## Werktijden
- Standaard werkweek: 40 uur (maandag t/m vrijdag)
- Flexibele werktijden tussen 07:00 en 19:00
- Thuiswerken: maximaal 2 dagen per week

## Vakantiedagen
- Wettelijke vakantiedagen: 20 dagen per jaar
- Bovenwettelijke vakantiedagen: 5 dagen per jaar
- Feestdagen volgens Nederlandse kalender

## Verlof
- Zwangerschapsverlof: 16 weken
- Partnerverlof: 5 dagen betaald + 5 weken aanvullend (70%)
- Calamiteitenverlof: naar behoefte
- Zorgverlof: 2 weken per jaar (70%)

## Reiskostenvergoeding
- OV-kosten: 100% vergoed
- Auto: €0,23 per km (max 50 km enkele reis)
- Fiets: €0,10 per km

## Pensioen
- Pensioenfonds: ABP/PFZW
- Werkgeversbijdrage: 2/3 van de premie
- Pensioenleeftijd: 67 jaar
"""

print("📄 Nederlandse documenten gemaakt:")
print(f"   - AVG Compliance Handleiding: {len(avg_document)} karakters")
print(f"   - Personeelshandboek: {len(hr_document)} karakters")

## 🤖 Stap 2: Maak een Nederlandse Assistent

Laten we een assistent maken die deze documenten kan doorzoeken:

In [ ]:
# Maak de assistent
assistant = client.beta.assistants.create(
    name="Nederlandse Bedrijfsassistent",
    instructions="""Je bent een behulpzame assistent voor Nederlandse bedrijven.
    
Je expertise omvat:
- AVG/GDPR compliance voor Nederlandse organisaties
- Nederlands arbeidsrecht en personeelszaken
- Nederlandse bedrijfsvoering en regelgeving

Antwoord altijd in het Nederlands. Verwijs naar specifieke artikelen of secties uit de documenten waar relevant.
Geef praktisch advies dat toepasbaar is voor Nederlandse situaties.""",
    model=deployment,
    tools=[{"type": "file_search"}]
)

print(f"🤖 Assistent aangemaakt: {assistant.name}")
print(f"   ID: {assistant.id}")

In [ ]:
# Maak een Vector Store
vector_store = client.beta.vector_stores.create(
    name="Nederlandse Bedrijfsdocumenten"
)

print(f"📚 Vector Store aangemaakt: {vector_store.name}")
print(f"   ID: {vector_store.id}")

In [ ]:
# Upload documenten
import tempfile

# Schrijf documenten naar tijdelijke bestanden
with tempfile.NamedTemporaryFile(mode='w', suffix='.txt', delete=False) as f:
    f.write(avg_document)
    avg_file_path = f.name

with tempfile.NamedTemporaryFile(mode='w', suffix='.txt', delete=False) as f:
    f.write(hr_document)
    hr_file_path = f.name

# Upload bestanden
with open(avg_file_path, 'rb') as f:
    avg_file = client.files.create(file=f, purpose="assistants")

with open(hr_file_path, 'rb') as f:
    hr_file = client.files.create(file=f, purpose="assistants")

print(f"📤 Bestanden geüpload:")
print(f"   - AVG document: {avg_file.id}")
print(f"   - HR document: {hr_file.id}")

In [ ]:
# Voeg bestanden toe aan Vector Store
batch = client.beta.vector_stores.file_batches.create_and_poll(
    vector_store_id=vector_store.id,
    file_ids=[avg_file.id, hr_file.id]
)

print(f"✅ Bestanden toegevoegd aan Vector Store")
print(f"   Status: {batch.status}")
print(f"   Bestanden verwerkt: {batch.file_counts.completed}")

In [ ]:
# Update assistent met Vector Store
assistant = client.beta.assistants.update(
    assistant_id=assistant.id,
    tool_resources={"file_search": {"vector_store_ids": [vector_store.id]}}
)

print(f"🔗 Assistent gekoppeld aan Vector Store")

## 💬 Stap 3: Test de Assistent met Nederlandse Vragen

In [ ]:
def ask_assistant(question: str):
    """Stel een vraag aan de assistent"""
    
    print(f"❓ Vraag: {question}")
    print("=" * 60)
    
    # Maak een thread
    thread = client.beta.threads.create()
    
    # Voeg bericht toe
    client.beta.threads.messages.create(
        thread_id=thread.id,
        role="user",
        content=question
    )
    
    # Run de assistent
    run = client.beta.threads.runs.create_and_poll(
        thread_id=thread.id,
        assistant_id=assistant.id
    )
    
    # Haal antwoord op
    messages = client.beta.threads.messages.list(thread_id=thread.id)
    
    print("\n🤖 Antwoord:")
    print("-" * 40)
    for msg in messages.data:
        if msg.role == "assistant":
            for content in msg.content:
                if hasattr(content, 'text'):
                    print(content.text.value)
    
    return messages

# Test met Nederlandse vragen
ask_assistant("Hoeveel vakantiedagen heb ik recht op per jaar?")

In [ ]:
print("\n" + "="*80 + "\n")
ask_assistant("Wat zijn de belangrijkste rechten van betrokkenen onder de AVG?")

In [ ]:
print("\n" + "="*80 + "\n")
ask_assistant("Hoeveel partnerverlof krijg ik als mijn partner bevalt?")

## 🧹 Opruimen

In [ ]:
# Ruim resources op
import os as os_module

try:
    client.beta.assistants.delete(assistant.id)
    print(f"🗑️ Assistent verwijderd")
except:
    pass

try:
    client.beta.vector_stores.delete(vector_store.id)
    print(f"🗑️ Vector Store verwijderd")
except:
    pass

try:
    os_module.remove(avg_file_path)
    os_module.remove(hr_file_path)
    print(f"🗑️ Tijdelijke bestanden verwijderd")
except:
    pass

print("\n✅ Opruimen voltooid")

## 🎓 Wat Je Hebt Geleerd

✅ **Assistants API**: Een assistent maken en configureren  
✅ **Vector Stores**: Documenten indexeren voor zoeken  
✅ **File Search**: Antwoorden baseren op documentinhoud  
✅ **Nederlandse Context**: AVG en HR documenten doorzoeken  

## 🚀 Volgende Stappen

- **06_complete_rag_pipeline_deep_dive.nl.ipynb** - Bouw je eigen RAG systeem!